# Lab 15: Basic Concepts of Mathematical Morphology

**Computer Vision Course**

Today you'll implement the fundamental morphological operations **from scratch** and then apply them — along with any tool from previous labs — to clean up noisy images and decode captchas.

**What you'll learn:**
- How erosion and dilation work at the pixel level
- The morphological gradient as a border detector
- Choosing the right operation for each type of noise
- Combining morphological and non-morphological tools creatively

**What you'll build:**
- `erosion()` — implemented without OpenCV
- `dilation()` — implemented without OpenCV
- `gradient()` — built from the two above
- Solutions to 3 noise challenges and 4 captcha challenges

**Connection to previous labs:**
- Lab 7: Convolution and smoothing kernels — the structuring element plays a similar role to the filter kernel
- Lab 13: Non-linear operations — morphology is non-linear too, and today you'll see why that matters for noise

## Setup

In [ ]:
"""
Computer Vision Course - Lab 15: Mathematical Morphology

This cell sets up the environment.
Works automatically for both local and Google Colab!
"""

import os
import sys

# Detect environment
IN_COLAB = 'google.colab' in sys.modules

print("=" * 60)
print("Computer Vision - Lab 15: Mathematical Morphology")
print("=" * 60)

if IN_COLAB:
    print("\n🔵 Running on Google Colab")
    print("-" * 60)

    if not os.path.exists('computer-vision'):
        print("📥 Cloning repository...")
        !git clone https://github.com/mjck/computer-vision.git
        print("✓ Repository cloned successfully")
    else:
        print("✓ Repository already exists")
        !git -C computer-vision pull origin main
        print("✓ Repository updated successfully")

    %cd computer-vision/labs/lab15_morphology
    print(f"✓ Current directory: {os.getcwd()}")

    sys.path.insert(0, '/content/computer-vision')
    print("✓ Python path configured")

    print("-" * 60)
    print("🟢 Colab setup complete!\n")

else:
    print("\n🟢 Running locally")
    print("-" * 60)
    print(f"✓ Current directory: {os.getcwd()}")

    repo_root = os.path.abspath('../..')
    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)
    print(f"✓ Repository root: {repo_root}")

    print("-" * 60)
    print("🟢 Local setup complete!\n")

print("=" * 60)
print("✅ Environment ready!")
print("=" * 60)

## Import Libraries

In [ ]:
import numpy as np
import cv2

# Import course utilities
try:
    from sdx import cv_imread, cv_imshow, cv_grayread
    print("✓ sdx module loaded")
except ImportError as e:
    print(f"❌ Could not import sdx: {e}")
    raise

print("✓ All imports successful")

---
## Part 1: Binary Operators

Morphological operations work by sliding a **structuring element** (kernel)
over the image, much like a convolution kernel — but instead of computing a
weighted sum, they ask a binary question at each position.

The answer determines the output pixel value.

### Load the Logo Image

In [ ]:
logo = cv_grayread('logo.png')

print(f"Image shape: {logo.shape}")
print(f"Unique values: {np.unique(logo)}")  # Should be [0, 255] for binary image

cv_imshow(logo)
print("Binary logo — white pixels are the foreground")

### Define the Structuring Element

The structuring element (kernel) defines the **neighbourhood shape** used
at each pixel. Its origin is at the center.

Available shapes:
- `cv2.MORPH_RECT` — filled rectangle
- `cv2.MORPH_ELLIPSE` — filled ellipse / disk
- `cv2.MORPH_CROSS` — plus sign

In [ ]:
# cv2.MORPH_RECT
# cv2.MORPH_ELLIPSE
# cv2.MORPH_CROSS

kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))

print("Structuring element (7×7 ellipse):")
print(kernel)
print()
print("1 = part of the SE,  0 = ignored position")

---
## 📝 Activity 1: Binary Erosion (3 points)

**The rule:** For each pixel `(y, x)`, the output is:
- `255` if **all** pixels in the kernel's neighbourhood that are mapped to `1` are `255`
- `0` otherwise

In other words: the kernel must **fit entirely inside** the foreground.

**Requirements:**
- Implement without using any OpenCV morphological functions
- Use NumPy as efficiently as possible — avoid unnecessary loops
- For simplicity, you may ignore border pixels (leave them as `0`)

**Hint:** For each pixel position, extract the neighbourhood window, apply the
kernel as a mask, and check whether all masked pixels equal 255.

In [ ]:
# ── Activity 1: Your code here ────────────────────────────────────────────

def erosion(input, kernel):
    height, width = input.shape
    output = np.zeros((height, width))

    # write your code here

    return output

# ──────────────────────────────────────────────────────────────────────────

cv_imshow(erosion(logo, kernel))
print("Eroded logo — object should be smaller/thinner")

**What to observe:**
- How did the shape change?
- What happened to thin strokes?
- Try different kernel sizes (3×3, 5×5, 11×11) and shapes — what changes?

---
## 📝 Activity 2: Binary Dilation (3 points)

**The rule:** For each pixel `(y, x)`, the output is:
- `255` if **at least one** pixel in the kernel's neighbourhood that is mapped to `1` is `255`
- `0` otherwise

In other words: the kernel only needs to **touch** the foreground somewhere.

**Requirements:**
- Implement without using any OpenCV morphological functions
- For simplicity, you may ignore border pixels (leave them as `0`)

**Hint:** The logic is the same as erosion but with `any` instead of `all`.

In [ ]:
# ── Activity 2: Your code here ────────────────────────────────────────────

def dilation(input, kernel):
    height, width = input.shape
    output = np.zeros((height, width))

    # write your code here

    return output

# ──────────────────────────────────────────────────────────────────────────

cv_imshow(dilation(logo, kernel))
print("Dilated logo — object should be larger/thicker")

**What to observe:**
- How did the shape change compared to erosion?
- What is the relationship between erosion and dilation?
- Apply erosion then dilation (opening) — does the shape return to its original size?

---
## 📝 Activity 3: Morphological Gradient (2 points)

**The idea:** The morphological gradient detects borders by computing:

```
gradient = dilation(image) − erosion(image)
```

Pixels that belong to the object interior are present in **both** dilation and erosion,
so they cancel out. Pixels on the **border** survive — they are in the dilation
but not in the erosion.

**Requirements:**
- Use your `erosion()` and `dilation()` functions from Activities 1 and 2
- If either of yours is not working correctly, you may use `cv2.erode()` and `cv2.dilate()` as a fallback
- The result should be a thin ring of white pixels tracing the object boundary

In [ ]:
# ── Activity 3: Your code here ────────────────────────────────────────────

def gradient(input, kernel):
    return input.copy()  # replace this line with your code

# ──────────────────────────────────────────────────────────────────────────

cv_imshow(gradient(logo, kernel))
print("Morphological gradient — should show a thin border around the object")

**What to observe:**
- Is the border one pixel wide?
- What happens with a larger kernel?
- How does this compare to Sobel or Laplacian on the same binary image (Lab 8)?

---
## Part 2: Noise Challenges

For each image below, use erosions and/or dilations to eliminate as much noise as
possible while preserving the original shapes.

**The constraint:** the resulting characters must **not** be noticeably thinner or
thicker than the originals — only the noise should be removed.

You may use `cv2.erode()`, `cv2.dilate()`, or `cv2.morphologyEx()` freely here.
You may also use your own `erosion()` and `dilation()` implementations.

**Think about:** What type of noise is it? How large are the noise pixels?
Choose your kernel size and sequence of operations accordingly.

### Salt Noise

Salt noise = isolated **white** pixels scattered over the image.

**Hint:** What operation removes isolated bright pixels?

In [ ]:
input_s = cv_grayread('logo-salt.png')

print("Noisy input (salt):")
cv_imshow(input_s)

# ── Your code here ──────────────────────────────────────────────────────────

output_s = input_s.copy()  # replace this line with your code

# ────────────────────────────────────────────────────────────────────────────

print("\nYour result:")
cv_imshow(output_s)

### Pepper Noise

Pepper noise = isolated **black** pixels scattered over the image.

**Hint:** Pepper is the complement of salt. What operation fills isolated dark pixels?

In [ ]:
input_p = cv_grayread('logo-pepper.png')

print("Noisy input (pepper):")
cv_imshow(input_p)

# ── Your code here ──────────────────────────────────────────────────────────

output_p = input_p.copy()  # replace this line with your code

# ────────────────────────────────────────────────────────────────────────────

print("\nYour result:")
cv_imshow(output_p)

### Salt-and-Pepper Noise

Both types of noise at once.

**Hint:** You'll need to chain two operations. Order matters!

In [ ]:
input_sp = cv_grayread('logo-salt-pepper.png')

print("Noisy input (salt and pepper):")
cv_imshow(input_sp)

# ── Your code here ──────────────────────────────────────────────────────────

output_sp = input_sp.copy()  # replace this line with your code

# ────────────────────────────────────────────────────────────────────────────

print("\nYour result:")
cv_imshow(output_sp)

---
## Part 3: Captcha Challenges

For each image below, use **any combination of techniques from previous labs**
to eliminate as much noise as possible while preserving the original characters.

**The constraint:** the resulting characters must not be significantly thinner or
thicker than the originals.

**You have the full toolkit:**
- Morphological operations (erosion, dilation, opening, closing, gradient)
- Smoothing filters (Gaussian, median)
- Thresholding
- Any combination you can justify

Each captcha has a different noise pattern — there is no single solution.
Explore, experiment, and explain your reasoning in a comment.

### Captcha A

In [ ]:
input_a = cv_grayread('captchaA.png')

print("Captcha A:")
cv_imshow(input_a)

# ── Your code here ──────────────────────────────────────────────────────────
# What type of noise do you see?
# What operations will you apply, and in what order?

output_a = input_a.copy()  # replace this line with your code

# ────────────────────────────────────────────────────────────────────────────

print("\nYour result:")
cv_imshow(output_a)

### Captcha B

In [ ]:
input_b = cv_grayread('captchaB.png')

print("Captcha B:")
cv_imshow(input_b)

# ── Your code here ──────────────────────────────────────────────────────────

output_b = input_b.copy()  # replace this line with your code

# ────────────────────────────────────────────────────────────────────────────

print("\nYour result:")
cv_imshow(output_b)

### Captcha C

In [ ]:
input_c = cv_grayread('captchaC.png')

print("Captcha C:")
cv_imshow(input_c)

# ── Your code here ──────────────────────────────────────────────────────────

output_c = input_c.copy()  # replace this line with your code

# ────────────────────────────────────────────────────────────────────────────

print("\nYour result:")
cv_imshow(output_c)

### Captcha D

In [ ]:
input_d = cv_grayread('captchaD.png')

print("Captcha D:")
cv_imshow(input_d)

# ── Your code here ──────────────────────────────────────────────────────────

output_d = input_d.copy()  # replace this line with your code

# ────────────────────────────────────────────────────────────────────────────

print("\nYour result:")
cv_imshow(output_d)

---
## Appendix: Grayscale (Level) Operators

Everything above used **binary** images (pixels are either 0 or 255).
Morphological operations also work on **grayscale** images — at each position
the output is the minimum (erosion) or maximum (dilation) intensity inside
the kernel, rather than all-or-nothing.

This section demonstrates the effect on a topographic map.

### Load the Map Image

In [ ]:
map_image = cv_grayread('map.png')

print(f"Map shape: {map_image.shape}")
print(f"Intensity range: [{map_image.min()}, {map_image.max()}]")

cv_imshow(map_image)
print("Grayscale map — intensity represents elevation")

### Define a Small Structuring Element

In [ ]:
level_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))

print("Structuring element (3×3 ellipse):")
print(level_kernel)

### Level Erosion

On a grayscale image, erosion replaces each pixel with the **minimum**
intensity found inside the kernel neighbourhood.

Effect: dark regions expand, bright regions shrink — valleys widen, peaks narrow.

In [ ]:
eroded_map = cv2.erode(map_image, level_kernel)

cv_imshow(eroded_map)
print("Level erosion — dark valleys have expanded")

### Level Dilation

On a grayscale image, dilation replaces each pixel with the **maximum**
intensity found inside the kernel neighbourhood.

Effect: bright regions expand, dark regions shrink — peaks widen, valleys narrow.

In [ ]:
dilated_map = cv2.dilate(map_image, level_kernel)

cv_imshow(dilated_map)
print("Level dilation — bright peaks have expanded")

### Compare All Three

In [ ]:
print("Original map:")
cv_imshow(map_image)

print("\nEroded (dark regions expand):")
cv_imshow(eroded_map)

print("\nDilated (bright regions expand):")
cv_imshow(dilated_map)

print("\n🤔 Notice the dual behaviour:")
print("   Erosion and dilation are opposites.")
print("   What erosion does to bright regions, dilation does to dark regions.")

---
## Summary

### What You Learned

1. **Erosion** — the structuring element must fit entirely inside the foreground
   - Shrinks objects, removes protrusions smaller than the kernel
   - Binary: `all` masked pixels must be 255
   - Grayscale: replace with neighbourhood minimum

2. **Dilation** — the structuring element needs to touch the foreground anywhere
   - Grows objects, fills holes smaller than the kernel
   - Binary: `any` masked pixel being 255 is enough
   - Grayscale: replace with neighbourhood maximum

3. **Morphological Gradient** — dilation − erosion
   - Extracts the boundary ring
   - Non-linear, exact on binary images

4. **Noise removal with morphology**
   - Salt → opening (erosion then dilation)
   - Pepper → closing (dilation then erosion)
   - Salt-and-pepper → opening then closing
   - Kernel size must exceed noise grain, must not exceed object features

5. **Creative application**
   - Captchas require combining multiple tools
   - Gaussian/median for continuous noise, morphology for structured noise
   - Order of operations matters

### Operation Cheat Sheet

| Operation | Binary rule | Grayscale | Removes |
|---|---|---|---|
| Erosion | all masked = 255 | min of window | Small bright, thin strokes |
| Dilation | any masked = 255 | max of window | Small dark, thin gaps |
| Opening = Ero→Dil | — | — | Salt noise, protrusions |
| Closing = Dil→Ero | — | — | Pepper noise, holes |
| Gradient = Dil−Ero | — | — | Everything except boundary |

### Connection to Deep Learning

Max-pooling in CNNs is a learned form of morphological dilation — it
takes the maximum value in a neighbourhood. This is why CNNs naturally
develop spatial invariance and detect the strongest activations within a region.

### ✏️ Final Reflection (1 point)

Answer these questions:

1. **Why does erosion shrink bright objects while dilation grows them?**

2. **For the salt noise challenge, why does opening work but closing would not?**

3. **Why did the captcha challenges require combinations of operations rather than a single morphological step?**

4. **What is the relationship between erosion/dilation on binary images and min/max pooling in a CNN?**

In [ ]:
# Your reflection:

# 1. Why erosion shrinks, dilation grows?
#    ...

# 2. Why opening works for salt noise but not closing?
#    ...

# 3. Why captchas need combinations?
#    ...

# 4. Relationship to CNN min/max pooling?
#    ...
